# Track B - Classical machine learning

This notebook compares TF-IDF + Naive Bayes, logistic regression, linear SVM, and random forest. Model selection uses only duplicate-safe grouped folds from the official training split. The official test split is evaluated once, after selection.

In [ ]:
from pathlib import Path

import pandas as pd

from ai4se.classical import (
    AVAILABLE_MODELS,
    TfidfConfig,
    make_classical_estimator_factory,
    model_display_name,
)
from ai4se.evaluation import (
    evaluate_cross_validation,
    evaluate_holdout_by_repository,
    save_evaluation,
)
from ai4se.reporting import result_rows, write_result_tables
from ai4se.service import IssueDataService

## Configuration

Word and character TF-IDF are combined. Full cleaning is appropriate for the bag-of-words representation; the title is repeated twice to retain its short, high-value signal.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RANDOM_STATE = 42
N_SPLITS = 5
CLEANING_LEVEL = "full"
MAX_WORDS = 1000
TITLE_WEIGHT = 2
OUTPUT_DIRECTORY = PROJECT_ROOT / "results" / "classical"

tfidf = TfidfConfig()
service = IssueDataService.from_loader(kind="memory")
train = service.prepare(
    "train",
    level=CLEANING_LEVEL,
    max_words=MAX_WORDS,
    title_weight=TITLE_WEIGHT,
)
len(train), tfidf

## Duplicate-safe model comparison

The TF-IDF vocabulary and IDF weights are learned inside each fold because vectorisation is part of each model pipeline.

In [ ]:
cv_results = {}
for model in AVAILABLE_MODELS:
    result = evaluate_cross_validation(
        train,
        make_classical_estimator_factory(
            model, tfidf=tfidf, random_state=RANDOM_STATE
        ),
        model_name=model_display_name(model),
        n_splits=N_SPLITS,
        random_state=RANDOM_STATE,
        metadata={
            "track": "B",
            "cleaning_level": CLEANING_LEVEL,
            "max_words": MAX_WORDS,
            "title_weight": TITLE_WEIGHT,
            "tfidf": tfidf.to_dict(),
        },
    )
    cv_results[model] = result
    save_evaluation(result, OUTPUT_DIRECTORY / f"{model}-cross-validation.json")

In [ ]:
ranking = pd.DataFrame(
    [
        {
            "model": model,
            "cross_repository_macro_f1": result["overall"]
            ["repository_average"]["macro_average"]["f1"],
        }
        for model, result in cv_results.items()
    ]
).sort_values("cross_repository_macro_f1", ascending=False)
ranking

## Final official evaluation

Only the cross-validation winner is now trained on each repository's complete training subset and evaluated on that repository's official test subset.

In [ ]:
selected_model = ranking.iloc[0]["model"]
test = service.prepare(
    "test",
    level=CLEANING_LEVEL,
    max_words=MAX_WORDS,
    title_weight=TITLE_WEIGHT,
)
official_result = evaluate_holdout_by_repository(
    train,
    test,
    make_classical_estimator_factory(
        selected_model, tfidf=tfidf, random_state=RANDOM_STATE
    ),
    model_name=model_display_name(selected_model),
    random_state=RANDOM_STATE,
    metadata={"track": "B", "selected_by_cross_validation": True},
)
save_evaluation(
    official_result,
    OUTPUT_DIRECTORY / f"{selected_model}-official-holdout.json",
)
official_result["overall"]["repository_average"]["macro_average"]

In [ ]:
all_results = [*cv_results.values(), official_result]
rows = [row for result in all_results for row in result_rows(result)]
write_result_tables(
    rows, PROJECT_ROOT / "results" / "tables", stem="classical_ml"
)